# Course 7: Building a Small Bangla Language Model
## Session 8: Evaluation II, tasks and benchmarks

**Goal:** fine-tune the model as a classifier, compare with a baseline, and score with macro-F1.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every code cell here was run once on a CPU (November 2026) using the small built-in sample. Cells marked TODO or NOT RUN need your data, a GPU or the network. The toy task stands in for a real Bangla sentiment dataset, and the BanglaBERT comparison was not run.

In [ ]:
SAMPLE_LINES = [
    "আমি বাংলায় গান গাই।",
    "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।",
    "বাংলা আমার মাতৃভাষা।",
    "আজ আকাশ খুব পরিষ্কার।",
    "ছোট মেয়েটি নদীর ধারে খেলছিল।",
    "বাবা বাজার থেকে মাছ কিনে আনলেন।",
    "মা রান্নাঘরে ভাত রান্না করছেন।",
    "আমরা সবাই মিলে স্কুলে যাই।",
    "বৃষ্টি হলে মাঠে পানি জমে।",
    "পাখিরা সকালে গান গায়।",
    "দাদু প্রতিদিন সন্ধ্যায় গল্প বলেন।",
    "ছেলেটি বই পড়তে খুব ভালোবাসে।",
    "নদীতে অনেক নৌকা চলছে।",
    "শীতকালে আমরা পিঠা খাই।",
    "ঢাকা বাংলাদেশের রাজধানী।",
    "আমার বন্ধু ভালো ফুটবল খেলে।",
]

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 4
    d_model: int = 256
    dropout: float = 0.0

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_head = c.n_head
        self.ln1 = nn.LayerNorm(c.d_model)
        self.qkv = nn.Linear(c.d_model, 3 * c.d_model)
        self.proj = nn.Linear(c.d_model, c.d_model)
        self.ln2 = nn.LayerNorm(c.d_model)
        self.fc = nn.Linear(c.d_model, 4 * c.d_model)
        self.out = nn.Linear(4 * c.d_model, c.d_model)
        self.drop = nn.Dropout(c.dropout)

    def attn(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.out(F.gelu(self.fc(self.ln2(x)))))
        return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab_size, c.d_model)
        self.pos = nn.Embedding(c.block_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln = nn.LayerNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, return_hidden=False):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        h = self.ln(x)
        if return_hidden:
            return h
        logits = self.head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.9):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.c.block_size:])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            sp, si = torch.sort(probs, descending=True)
            keep = torch.cumsum(sp, dim=-1) - sp < top_p
            sp = sp * keep
            sp = sp / sp.sum(dim=-1, keepdim=True)
            nxt = si.gather(-1, torch.multinomial(sp, 1))
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers

def train_bpe(lines, vocab_size, specials=("[UNK]", "<|endoftext|>")):
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.normalizer = normalizers.NFC()
    tok.pre_tokenizer = pre_tokenizers.Metaspace()
    tok.decoder = decoders.Metaspace()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=list(specials))
    tok.train_from_iterator(lines, trainer)
    return tok

In [ ]:
# A tiny model and tokenizer trained on the built-in sample, so this notebook runs anywhere.
# Replace with your own tokenizer file and checkpoint from sessions 4 and 6.
torch.manual_seed(0)
tok = train_bpe(SAMPLE_LINES * 20, 300)
EOT = tok.token_to_id("<|endoftext|>")
ids = []
for line in SAMPLE_LINES * 20:
    ids += tok.encode(line).ids + [EOT]
data = torch.tensor(ids)
cfg = Config(vocab_size=tok.get_vocab_size(), block_size=32, n_layer=2, n_head=2, d_model=64)
model = GPT(cfg)

def get_batch(data, block_size, batch_size):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for step in range(300):
    x, y = get_batch(data, cfg.block_size, 32)
    _, loss = model(x, y)
    opt.zero_grad(); loss.backward(); opt.step()
print("tiny model trained, final loss", round(loss.item(), 3))

### 1. A toy labelled task
Replace with a real Bangla sentiment dataset and its standard split. The toy labels mark sentences that mention family members.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
family = ["বাবা", "মা", "দাদু", "মেয়ে", "ছেলে", "বন্ধু"]
texts = SAMPLE_LINES * 6
labels = [int(any(w in t for w in family)) for t in texts]
import random
random.seed(0)
idx = list(range(len(texts))); random.shuffle(idx)
cut = int(0.7 * len(idx))
train_idx, test_idx = idx[:cut], idx[cut:]
print("class balance:", sum(labels) / len(labels))

### 2. Majority-class baseline

In [ ]:
y_test = [labels[i] for i in test_idx]
majority = max(set(labels), key=labels.count)
base_pred = [majority] * len(y_test)
print("baseline accuracy:", round(accuracy_score(y_test, base_pred), 3),
      "| macro-F1:", round(f1_score(y_test, base_pred, average="macro"), 3))

### 3. Classification head on the last token's hidden state

In [ ]:
class Classifier(nn.Module):
    def __init__(self, gpt, n_classes):
        super().__init__()
        self.gpt = gpt
        self.head = nn.Linear(gpt.c.d_model, n_classes)
    def forward(self, ids):
        h = self.gpt(ids, return_hidden=True)      # (1, T, d)
        return self.head(h[:, -1, :])              # last position has seen the whole text

def encode(text):
    return torch.tensor([tok.encode(text).ids[:cfg.block_size]])

torch.manual_seed(0)
clf = Classifier(model, 2)
opt = torch.optim.AdamW(clf.parameters(), lr=1e-3)
clf.train()
for epoch in range(3):
    random.shuffle(train_idx)
    for i in train_idx:
        loss = F.cross_entropy(clf(encode(texts[i])), torch.tensor([labels[i]]))
        opt.zero_grad(); loss.backward(); opt.step()
clf.eval()
with torch.no_grad():
    pred = [clf(encode(texts[i])).argmax(-1).item() for i in test_idx]
print("model accuracy:", round(accuracy_score(y_test, pred), 3),
      "| macro-F1:", round(f1_score(y_test, pred, average="macro"), 3))

The toy score is near perfect because the same sentences repeat in training and test. On a real dataset, expect a result well above the baseline and below BanglaBERT.

### 4. Multiple choice by log-probability

In [ ]:
@torch.no_grad()
def log_prob(model, context, option):
    c_ids = tok.encode(context).ids; o_ids = tok.encode(" " + option).ids
    ids = torch.tensor([c_ids + o_ids])
    logits, _ = model(ids[:, :-1])
    lp = torch.log_softmax(logits[0], dim=-1)
    targets = ids[0, 1:]
    return lp[range(len(targets)), targets][len(c_ids) - 1:].sum().item()

context = "আমার সোনার বাংলা, আমি তোমায়"
for option in ["ভালোবাসি।", "বাজার"]:
    print(option, round(log_prob(model, context, option), 2))

### 5. BanglaBERT comparison (TODO, NOT RUN)
Needs the network and a GPU. Use the csebuetnlp normalizer before tokenizing, the same data, a learning-rate search of the same size, and three seeds.

In [ ]:
# !pip install transformers git+https://github.com/csebuetnlp/normalizer
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# from normalizer import normalize
# bert_tok = AutoTokenizer.from_pretrained("csebuetnlp/banglabert")
# bert = AutoModelForSequenceClassification.from_pretrained("csebuetnlp/banglabert", num_labels=3)

### TODO
- Results table: baseline, your model, BanglaBERT, macro-F1 over three seeds, with parameter counts.
- Contamination check: search your pretraining corpus for 100 random test sentences.
- Read 30 errors and group them.